In [ ]:
import os
from functools import partial

import GPy
import numpy as np
from sklearn.metrics.pairwise import euclidean_distances

from matplotlib.lines import Line2D
from matplotlib import pyplot as plt

from BO_loop import BO_loop_RS
from BO_loop import BO_loop_RO
from adv_RO import find_x_ro_action
from adv_RS1 import find_x_rs_action_RS1
from adv_RS2 import find_x_rs_action_RS2

In [3]:
def f_poly(x, y):
    """
    Computes the value of the polynomial function f_poly(x, y) similar to the one in Bogunovic et. al.

    Domain:
    The function is defined on a uniformly spaced grid of points in the range:
    x ∈ (-0.95, 3.2)
    y ∈ (-0.45, 4.4)

    Parameters:
    x (float or np.array): Array of input coordinates
    y (float or np.array): Array of input coordinates
    
    Returns:
    float or np.array: The computed value of the polynomial at the given (x, y) coordinates.
    """
    return (-2 * x**6 + 12.2 * x**5 - 21.2 * x**4 - 6.2 * x + 6.4 * x**3 + 4.7 * x**2 
            - y**6 + 11 * y**5 - 43.3 * y**4 + 10 * y + 74.8 * y**3 - 56.9 * y**2 
            + 4.1 * x * y + 0.05 * y**2 * x**2 - 0.4 * y**2 * x - 0.4 * x**2 * y)

In [ ]:
distance_metric = partial(euclidean_distances, squared=True)

# Define the range and resolution
x_min, x_max = -0.95, 3.2
y_min, y_max = -0.45, 4.4
resolution = 32  # Number of points along each axis

# Create arrays of x and y values
x = np.linspace(x_min, x_max, resolution)
y = np.linspace(y_min, y_max, resolution)

# Create a meshgrid from x and y arrays
X_grid, Y_grid = np.meshgrid(x, y)
X = np.vstack([X_grid.ravel(), Y_grid.ravel()]).T
Z_grid = f_poly(X_grid, Y_grid)
Z = f_poly(X[:, 0], X[:, 1])

tau = -10
epsilon_t = 0.5
r_ro_true = epsilon_t * 1
r_ro_large = epsilon_t * 4
r_ro_small = epsilon_t * 0.5
attack_mode = 0

index_x_rs1, kappa_rs1 = find_x_rs_action_RS1(X=X, Y=Z, tau=tau, dist_metric=euclidean_distances, pow=1)
index_x_rs2, epsilon = find_x_rs_action_RS2(X=X, Y=Z, tau=tau, dist_metric=distance_metric)
epsilon = np.sqrt(epsilon)
index_x_rsg, kappa_rsg = find_x_rs_action_RS1(X=X, Y=Z, tau=tau, dist_metric=euclidean_distances, pow=2)
index_x_ro_true = find_x_ro_action(X=X, Y=Z, r_ro=r_ro_true**2, dist_metric=distance_metric)
index_x_ro_large = find_x_ro_action(X=X, Y=Z, r_ro=r_ro_large**2, dist_metric=distance_metric)
index_x_ro_small = find_x_ro_action(X=X, Y=Z, r_ro=r_ro_small**2, dist_metric=distance_metric)

print(f"RS1: {X[index_x_rs1]}, kappa: {kappa_rs1}")
print(f"RS2: {X[index_x_rs2]}, epsilon: {epsilon}")
print(f"RSG: {X[index_x_rsg]}, kappa_rsg: {kappa_rsg}")
print(f"RO True: {X[index_x_ro_true]}")
print(f"RO Large: {X[index_x_ro_large]}")
print(f"RO Small: {X[index_x_ro_small]}")

In [ ]:
# Set the figure size
plt.figure(figsize=(10, 6))

# Plot the filled contours
plt.contourf(X_grid, Y_grid, Z_grid, levels=20, cmap='plasma')
plt.colorbar(label='f(X, Y)')
plt.xlabel('x')
plt.ylabel('y')

# Highlight the specific contour corresponding to tau
contour = plt.contour(X_grid, Y_grid, Z_grid, levels=[tau], colors='black', linestyles='dashed')

# Scatter plots for the points
plt.scatter(X[index_x_rs1, 0], X[index_x_rs1, 1], c='r', marker='^', label=r'RS1 $\tau=-10$', s=120)
plt.scatter(X[index_x_rs2, 0], X[index_x_rs2, 1], c='m', marker='^', label=r'RS2 $\tau=-10$', s=120)
plt.scatter(X[index_x_ro_true, 0], X[index_x_ro_true, 1], c='b', marker='d', label=rf'RO $r={r_ro_true}$', s=120)
plt.scatter(X[index_x_ro_large, 0], X[index_x_ro_large, 1], c='darkred', marker='d', label=rf'RO $r={r_ro_large}$', s=120)
plt.scatter(X[index_x_ro_small, 0], X[index_x_ro_small, 1], c='g', marker='d', label=rf'RO $r={r_ro_small}$', s=120)

# Add custom legend entry for the contour
custom_lines = [Line2D([0], [0], color='black', linestyle='dashed', label=r'$\tau=-10$')]

# Add legend outside the plot
plt.legend(handles=custom_lines + plt.gca().get_legend_handles_labels()[0], bbox_to_anchor=(1.2, 1), loc='upper left')

# Adjust layout to make room for the legend
plt.tight_layout(rect=[0, 0, 0.85, 1])
# Save as pdf
dir = 'figures'
os.makedirs(dir, exist_ok=True)
# plt.savefig(os.path.join(dir, 'opt_objectives_poly_func.pdf'), format='pdf', bbox_inches='tight')
# Show plot
plt.show()

In [ ]:
kern_poly = GPy.kern.Poly(2, order=6, scale=1.11, bias=2.84, variance=1.32)

tau = -10
epsilon_t = 0.5
r_ro_true = epsilon_t * 1
r_ro_large = epsilon_t * 4
r_ro_small = epsilon_t * 0.5
attack_mode = 0

T=100
distance_metric = partial(euclidean_distances, squared=True)
sigma_noise = 1
num_initial_points = 10
num_iter = 1

Z = Z.reshape(-1, 1)
print(Z.shape)

path = "output_arrays/poly_func_multiple/"
os.makedirs(path, exist_ok=True)
attacks = [0, 1, 2]

# import sys
# sys.exit("Comment out to run the BO loops")

SHOULD_SAVE = True

for attack in attacks:
    attack_mode = attack
    print(f'Attack mode: {attack_mode}')
    print(f'Num_iter: {num_iter}')
    
    print('Starting RS1 loop')
    history_RS1 = BO_loop_RS(X=X, Y=Z, acquisition_function=find_x_rs_action_RS1, kernel=kern_poly, sigma_noise=sigma_noise,
                            num_init_samples=num_initial_points, distance_metric=euclidean_distances,
                            attack_mode=attack_mode, tau=tau, epsilon_t=epsilon_t, T=T, print_samples=False, num_iter=num_iter) 
    if SHOULD_SAVE:
        array_RS1_name = f'f_poly_N_{num_iter}_tau_{tau}_epsilon_{epsilon_t}_T_{T}_attack_{attack_mode}_RS1.npy'
        np.save(os.path.join(path, array_RS1_name), history_RS1)
    
    print('Starting RS2 loop')
    history_RS2 = BO_loop_RS(X=X, Y=Z, acquisition_function=find_x_rs_action_RS2, kernel=kern_poly, sigma_noise=sigma_noise,
                            num_init_samples=num_initial_points, distance_metric=distance_metric,
                            attack_mode=attack_mode, tau=tau, epsilon_t=epsilon_t**2, T=T, print_samples=False, num_iter=num_iter) 
    if SHOULD_SAVE:
        array_RS2_name = f'f_poly_N_{num_iter}_tau_{tau}_epsilon_{epsilon_t}_T_{T}_attack_{attack_mode}_RS2.npy'
        np.save(os.path.join(path, array_RS2_name), history_RS2)
    
    print('Starting RSG loop')
    partial_acq_rsg = partial(find_x_rs_action_RS1, pow=2)
    history_RSG = BO_loop_RS(X=X, Y=Z, acquisition_function=partial_acq_rsg, kernel=kern_poly, sigma_noise=sigma_noise,
                            num_init_samples=num_initial_points, distance_metric=distance_metric,
                            attack_mode=attack_mode, tau=tau, epsilon_t=epsilon_t, T=T, print_samples=False, num_iter=num_iter) 
    if SHOULD_SAVE:
        array_RSG_name = f'f_poly_N_{num_iter}_tau_{tau}_epsilon_{epsilon_t}_T_{T}_attack_{attack_mode}_RSG_p2.npy'
        np.save(os.path.join(path, array_RSG_name), history_RSG)
    
    print('Starting RO_true loop')
    history_RO_true = BO_loop_RO(X=X, Y=Z, num_init_samples=num_initial_points,
                                kernel=kern_poly, distance_metric=distance_metric, sigma_noise=sigma_noise,
                            attack_mode=attack_mode, r_ro=epsilon_t**2, epsilon_t=epsilon_t**2, T=T, print_samples=False, num_iter=num_iter) 
    if SHOULD_SAVE:
        array_RO_true_name = f'f_poly_N_{num_iter}_r_{r_ro_true:.2f}_epsilon_{epsilon_t}_T_{T}_attack_{attack_mode}_RO.npy'
        np.save(os.path.join(path, array_RO_true_name), history_RO_true)
    
    print('Starting RO_small loop')
    history_RO_small = BO_loop_RO(X=X, Y=Z, num_init_samples=num_initial_points,
                                sigma_noise=sigma_noise, kernel=kern_poly, distance_metric=distance_metric,
                            attack_mode=attack_mode, r_ro=r_ro_small**2, epsilon_t=epsilon_t**2, T=T, print_samples=False, num_iter=num_iter) 
    if SHOULD_SAVE:
        array_RO_small_name = f'f_poly_N_{num_iter}_r_{r_ro_small:.2f}_epsilon_{epsilon_t}_T_{T}_attack_{attack_mode}_RO.npy'
        np.save(os.path.join(path, array_RO_small_name), history_RO_small)
    
    print('Starting RO_large loop')
    history_RO_large = BO_loop_RO(X=X, Y=Z, num_init_samples=num_initial_points, 
                                sigma_noise=sigma_noise, kernel=kern_poly, distance_metric=distance_metric,
                            attack_mode=attack_mode, r_ro=r_ro_large**2, epsilon_t=epsilon_t**2, T=T, print_samples=False, num_iter=num_iter) 
    if SHOULD_SAVE:
        array_RO_large_name = f'f_poly_N_{num_iter}_r_{r_ro_large:.2f}_epsilon_{epsilon_t}_T_{T}_attack_{attack_mode}_RO.npy'
        np.save(os.path.join(path, array_RO_large_name), history_RO_large)

In [6]:
# Load the data for plotting
epsilon_t = 0.5
r_ro_true = epsilon_t * 1
r_ro_large = epsilon_t * 4
r_ro_small = epsilon_t * 0.5

data_RS1_0 = np.load(os.path.join(path, f'f_poly_N_{num_iter}_tau_-10_epsilon_{epsilon_t}_T_100_attack_0_RS1.npy'), allow_pickle=True)
data_RS1_1 = np.load(os.path.join(path, f'f_poly_N_{num_iter}_tau_-10_epsilon_{epsilon_t}_T_100_attack_1_RS1.npy'), allow_pickle=True)
data_RS1_2 = np.load(os.path.join(path, f'f_poly_N_{num_iter}_tau_-10_epsilon_{epsilon_t}_T_100_attack_2_RS1.npy'), allow_pickle=True)
data_RS2_0 = np.load(os.path.join(path, f'f_poly_N_{num_iter}_tau_-10_epsilon_{epsilon_t}_T_100_attack_0_RS2.npy'), allow_pickle=True)
data_RS2_1 = np.load(os.path.join(path, f'f_poly_N_{num_iter}_tau_-10_epsilon_{epsilon_t}_T_100_attack_1_RS2.npy'), allow_pickle=True)
data_RS2_2 = np.load(os.path.join(path, f'f_poly_N_{num_iter}_tau_-10_epsilon_{epsilon_t}_T_100_attack_2_RS2.npy'), allow_pickle=True)
data_RSG_0 = np.load(os.path.join(path, f'f_poly_N_{num_iter}_tau_-10_epsilon_{epsilon_t}_T_100_attack_0_RSG_p2.npy'), allow_pickle=True)
data_RSG_1 = np.load(os.path.join(path, f'f_poly_N_{num_iter}_tau_-10_epsilon_{epsilon_t}_T_100_attack_1_RSG_p2.npy'), allow_pickle=True)
data_RSG_2 = np.load(os.path.join(path, f'f_poly_N_{num_iter}_tau_-10_epsilon_{epsilon_t}_T_100_attack_2_RSG_p2.npy'), allow_pickle=True)
data_RO_true_0 = np.load(os.path.join(path, f'f_poly_N_{num_iter}_r_{r_ro_true:.2f}_epsilon_{epsilon_t}_T_100_attack_0_RO.npy'), allow_pickle=True)
data_RO_true_1 = np.load(os.path.join(path, f'f_poly_N_{num_iter}_r_{r_ro_true:.2f}_epsilon_{epsilon_t}_T_100_attack_1_RO.npy'), allow_pickle=True)
data_RO_true_2 = np.load(os.path.join(path, f'f_poly_N_{num_iter}_r_{r_ro_true:.2f}_epsilon_{epsilon_t}_T_100_attack_2_RO.npy'), allow_pickle=True)
data_RO_large_0 = np.load(os.path.join(path, f'f_poly_N_{num_iter}_r_{r_ro_large:.2f}_epsilon_{epsilon_t}_T_100_attack_0_RO.npy'), allow_pickle=True)
data_RO_large_1 = np.load(os.path.join(path, f'f_poly_N_{num_iter}_r_{r_ro_large:.2f}_epsilon_{epsilon_t}_T_100_attack_1_RO.npy'), allow_pickle=True)
data_RO_large_2 = np.load(os.path.join(path, f'f_poly_N_{num_iter}_r_{r_ro_large:.2f}_epsilon_{epsilon_t}_T_100_attack_2_RO.npy'), allow_pickle=True)
data_RO_small_0 = np.load(os.path.join(path, f'f_poly_N_{num_iter}_r_{r_ro_small:.2f}_epsilon_{epsilon_t}_T_100_attack_0_RO.npy'), allow_pickle=True)
data_RO_small_1 = np.load(os.path.join(path, f'f_poly_N_{num_iter}_r_{r_ro_small:.2f}_epsilon_{epsilon_t}_T_100_attack_1_RO.npy'), allow_pickle=True)
data_RO_small_2 = np.load(os.path.join(path, f'f_poly_N_{num_iter}_r_{r_ro_small:.2f}_epsilon_{epsilon_t}_T_100_attack_2_RO.npy'), allow_pickle=True)

In [7]:
# Calculate Cumulative Regrets
cum_reg_list_0 = [
    np.mean(np.cumsum(np.maximum(tau - Z[data_RS1_0[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.mean(np.cumsum(np.maximum(tau - Z[data_RS2_0[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.mean(np.cumsum(np.maximum(tau - Z[data_RSG_0[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.mean(np.cumsum(np.maximum(tau - Z[data_RO_true_0[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.mean(np.cumsum(np.maximum(tau - Z[data_RO_small_0[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.mean(np.cumsum(np.maximum(tau - Z[data_RO_large_0[:,1,:].astype(int)], 0), axis=1), axis=0)
]
cum_reg_list_1 = [
    np.mean(np.cumsum(np.maximum(tau - Z[data_RS1_1[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.mean(np.cumsum(np.maximum(tau - Z[data_RS2_1[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.mean(np.cumsum(np.maximum(tau - Z[data_RSG_1[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.mean(np.cumsum(np.maximum(tau - Z[data_RO_true_1[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.mean(np.cumsum(np.maximum(tau - Z[data_RO_small_1[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.mean(np.cumsum(np.maximum(tau - Z[data_RO_large_1[:,1,:].astype(int)], 0), axis=1), axis=0)
]
cum_reg_list_2 = [
    np.mean(np.cumsum(np.maximum(tau - Z[data_RS1_2[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.mean(np.cumsum(np.maximum(tau - Z[data_RS2_2[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.mean(np.cumsum(np.maximum(tau - Z[data_RSG_2[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.mean(np.cumsum(np.maximum(tau - Z[data_RO_true_2[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.mean(np.cumsum(np.maximum(tau - Z[data_RO_small_2[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.mean(np.cumsum(np.maximum(tau - Z[data_RO_large_2[:,1,:].astype(int)], 0), axis=1), axis=0)
]

# Calculate Cumulative Regrets
cum_reg_std_list_0 = [
    np.std(np.cumsum(np.maximum(tau - Z[data_RS1_0[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.std(np.cumsum(np.maximum(tau - Z[data_RS2_0[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.std(np.cumsum(np.maximum(tau - Z[data_RSG_0[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.std(np.cumsum(np.maximum(tau - Z[data_RO_true_0[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.std(np.cumsum(np.maximum(tau - Z[data_RO_small_0[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.std(np.cumsum(np.maximum(tau - Z[data_RO_large_0[:,1,:].astype(int)], 0), axis=1), axis=0),
]
cum_reg_std_list_1 = [
    np.std(np.cumsum(np.maximum(tau - Z[data_RS1_1[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.std(np.cumsum(np.maximum(tau - Z[data_RS2_1[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.std(np.cumsum(np.maximum(tau - Z[data_RSG_1[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.std(np.cumsum(np.maximum(tau - Z[data_RO_true_1[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.std(np.cumsum(np.maximum(tau - Z[data_RO_small_1[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.std(np.cumsum(np.maximum(tau - Z[data_RO_large_1[:,1,:].astype(int)], 0), axis=1), axis=0),
]
cum_reg_std_list_2 = [
    np.std(np.cumsum(np.maximum(tau - Z[data_RS1_2[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.std(np.cumsum(np.maximum(tau - Z[data_RS2_2[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.std(np.cumsum(np.maximum(tau - Z[data_RSG_2[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.std(np.cumsum(np.maximum(tau - Z[data_RO_true_2[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.std(np.cumsum(np.maximum(tau - Z[data_RO_small_2[:,1,:].astype(int)], 0), axis=1), axis=0),
    np.std(np.cumsum(np.maximum(tau - Z[data_RO_large_2[:,1,:].astype(int)], 0), axis=1), axis=0),
]

In [ ]:
plt.style.use('seaborn-v0_8-paper')
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 16,
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})
# Ensure LaTeX is used for rendering text
plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'

fig, axs = plt.subplots(1, 3, figsize=(15, 3.5))
alpha_error = 0.5
markevery = 16

# Plotting cum_reg_list_0
axs[0].plot(cum_reg_list_0[0], label=r'RS-G, $p=1$ (RS-1)', c='r', linestyle='--', marker='^', markevery=markevery, markersize=8)
axs[0].errorbar(
    range(0, T, markevery), cum_reg_list_0[0][::markevery].flatten(),
    yerr=cum_reg_std_list_0[0][::markevery].flatten()*alpha_error,
    fmt='none', capsize=5, capthick=2, c='r',
)
axs[0].plot(cum_reg_list_0[1], label=r'RS-G, $p=\infty$ (RS-2)', c='m', linestyle='--', marker='^', markevery=markevery, markersize=8)
axs[0].errorbar(
    range(0, T, markevery), cum_reg_list_0[1][::markevery].flatten(),
    yerr=cum_reg_std_list_0[1][::markevery].flatten()*alpha_error,
    fmt='none', capsize=5, capthick=2, c='m',
)
axs[0].plot(cum_reg_list_0[2], label=r'RS-G, $p=2$', c='orange', linestyle='--', marker='^', markevery=markevery, markersize=8)
axs[0].errorbar(
    range(0, T, markevery), cum_reg_list_0[2][::markevery].flatten(),
    yerr=cum_reg_std_list_0[2][::markevery].flatten()*alpha_error,
    fmt='none', capsize=5, capthick=2, c='orange',
)
axs[0].plot(cum_reg_list_0[3], label=r'RO $r=\epsilon_t$', c='b', linestyle='--', marker='d', markevery=markevery, markersize=8)
axs[0].errorbar(
    range(0, T, markevery), cum_reg_list_0[3][::markevery].flatten(),
    yerr=cum_reg_std_list_0[3][::markevery].flatten()*alpha_error,
    fmt='none', capsize=5, capthick=2, c='b',
)
axs[0].plot(cum_reg_list_0[4], label=r'RO $r=0.5 \cdot \epsilon_t$', c='g', linestyle='--', marker='d', markevery=markevery, markersize=8)
axs[0].errorbar(
    range(0, T, markevery), cum_reg_list_0[4][::markevery].flatten(),
    yerr=cum_reg_std_list_0[4][::markevery].flatten()*alpha_error,
    fmt='none', capsize=5, capthick=2, c='g',
)
axs[0].plot(cum_reg_list_0[5], label=r'RO $r=4 \cdot \epsilon_t$', c='darkred', linestyle='--', marker='d', markevery=markevery, markersize=8)
axs[0].errorbar(
    range(0, T, markevery), cum_reg_list_0[5][::markevery].flatten(),
    yerr=cum_reg_std_list_0[5][::markevery].flatten()*alpha_error,
    fmt='none', capsize=5, capthick=2, c='darkred',
)
axs[0].set_xlabel('Round t')
axs[0].set_ylabel('Lenient Regret')
axs[0].set_title('Random Attack')

# Plotting cum_reg_list_1
axs[1].plot(cum_reg_list_1[0], label=r'RS-G, $p=1$ (RS-1)', c='r', linestyle='--', marker='^', markevery=markevery, markersize=8)
axs[1].errorbar(
    range(0, T, markevery), cum_reg_list_1[0][::markevery].flatten(),
    yerr=cum_reg_std_list_1[0][::markevery].flatten()*alpha_error,
    fmt='none', capsize=5, capthick=2, c='r',
)
axs[1].plot(cum_reg_list_1[1], label=r'RS-G, $p=\infty$ (RS-2)', c='m', linestyle='--', marker='^', markevery=markevery, markersize=8)
axs[1].errorbar(
    range(0, T, markevery), cum_reg_list_1[1][::markevery].flatten(),
    yerr=cum_reg_std_list_1[1][::markevery].flatten()*alpha_error,
    fmt='none', capsize=5, capthick=2, c='m',
)
axs[1].plot(cum_reg_list_1[2], label=r'RS-G, $p=2$', c='orange', linestyle='--', marker='^', markevery=markevery, markersize=8)
axs[1].errorbar(
    range(0, T, markevery), cum_reg_list_1[2][::markevery].flatten(),
    yerr=cum_reg_std_list_1[2][::markevery].flatten()*alpha_error,
    fmt='none', capsize=5, capthick=2, c='orange',
)
axs[1].plot(cum_reg_list_1[3], label=r'RO $r=\epsilon_t$', c='b', linestyle='--', marker='d', markevery=markevery, markersize=8)
axs[1].errorbar(
    range(0, T, markevery), cum_reg_list_1[3][::markevery].flatten(),
    yerr=cum_reg_std_list_1[3][::markevery].flatten()*alpha_error,
    fmt='none', capsize=5, capthick=2, c='b',
)
axs[1].plot(cum_reg_list_1[4], label=r'RO $r=0.5 \cdot \epsilon_t$', c='g', linestyle='--', marker='d', markevery=markevery, markersize=8)
axs[1].errorbar(
    range(0, T, markevery), cum_reg_list_1[4][::markevery].flatten(),
    yerr=cum_reg_std_list_1[4][::markevery].flatten()*alpha_error,
    fmt='none', capsize=5, capthick=2, c='g',
)
axs[1].plot(cum_reg_list_1[5], label=r'RO $r=4 \cdot \epsilon_t$', c='darkred', linestyle='--', marker='d', markevery=markevery, markersize=8)
axs[1].errorbar(
    range(0, T, markevery), cum_reg_list_1[5][::markevery].flatten(),
    yerr=cum_reg_std_list_1[5][::markevery].flatten()*alpha_error,
    fmt='none', capsize=5, capthick=2, c='darkred',
)
axs[1].set_xlabel('Round t')
axs[1].set_ylabel('Lenient Regret')
axs[1].set_title('LCB Attack')

# Plotting cum_reg_list_2
axs[2].plot(cum_reg_list_2[0], label=r'RS-G, $p=1$ (RS-1)', c='r', linestyle='--', marker='^', markevery=markevery, markersize=8)
axs[2].errorbar(
    range(0, T, markevery), cum_reg_list_2[0][::markevery].flatten(),
    yerr=cum_reg_std_list_2[0][::markevery].flatten()*alpha_error,
    fmt='none', capsize=5, capthick=2, c='r',
)
axs[2].plot(cum_reg_list_2[1], label=r'RS-G, $p=\infty$ (RS-2)', c='m', linestyle='--', marker='^', markevery=markevery, markersize=8)
axs[2].errorbar(
    range(0, T, markevery), cum_reg_list_2[1][::markevery].flatten(),
    yerr=cum_reg_std_list_2[1][::markevery].flatten()*alpha_error,
    fmt='none', capsize=5, capthick=2, c='m',
)
axs[2].plot(cum_reg_list_2[2], label=r'RS-G, $p=2$', c='orange', linestyle='--', marker='^', markevery=markevery, markersize=8)
axs[2].errorbar(
    range(0, T, markevery), cum_reg_list_2[2][::markevery].flatten(),
    yerr=cum_reg_std_list_2[2][::markevery].flatten()*alpha_error,
    fmt='none', capsize=5, capthick=2, c='orange',
)
axs[2].plot(cum_reg_list_2[3], label=r'RO $r=\epsilon_t$', c='b', linestyle='--', marker='d', markevery=markevery, markersize=8)
axs[2].errorbar(
    range(0, T, markevery), cum_reg_list_2[3][::markevery].flatten(),
    yerr=cum_reg_std_list_2[3][::markevery].flatten()*alpha_error,
    fmt='none', capsize=5, capthick=2, c='b',
)
axs[2].plot(cum_reg_list_2[4], label=r'RO $r=0.5 \cdot \epsilon_t$', c='g', linestyle='--', marker='d', markevery=markevery, markersize=8)
axs[2].errorbar(
    range(0, T, markevery), cum_reg_list_2[4][::markevery].flatten(),
    yerr=cum_reg_std_list_2[4][::markevery].flatten()*alpha_error,
    fmt='none', capsize=5, capthick=2, c='g',
)
axs[2].plot(cum_reg_list_2[5], label=r'RO $r=4 \cdot \epsilon_t$', c='darkred', linestyle='--', marker='d', markevery=markevery, markersize=8)
axs[2].errorbar(
    range(0, T, markevery), cum_reg_list_2[5][::markevery].flatten(),
    yerr=cum_reg_std_list_2[5][::markevery].flatten()*alpha_error,
    fmt='none', capsize=5, capthick=2, c='darkred',
)
axs[2].set_xlabel('Round t')
axs[2].set_ylabel('Lenient Regret')
axs[2].set_title('Worst Case Attack')
axs[2].set_ylim([-50, 2000])

# Adding legends for tau and epsilon_t
# axs[0].legend(title=rf'$\tau=-10$, $\epsilon_t={epsilon_t:.1f}$')
axs[1].legend(title=rf'$\tau=-10$, $\epsilon_t={epsilon_t:.1f}$')
# axs[2].legend(title=rf'$\tau=-10$, $\epsilon_t={epsilon_t:.1f}$')

plt.tight_layout()
# Save as pdf
dir = 'figures'
os.makedirs(dir, exist_ok=True)
# plt.savefig(os.path.join(dir, 'regret_GP_func_rsg_N_100.pdf'), format='pdf', bbox_inches='tight')
# Display the plot
plt.show()

In [9]:
# Calculate Cumulative Regrets
cum_rs_reg_list_0 = [
    np.cumsum(np.mean(np.maximum(tau - kappa_rs1*epsilon_t - Z[data_RS1_0[:,1,:].astype(int)], 0), axis=0)),
    np.cumsum(np.mean(np.maximum(tau - kappa_rs1*epsilon_t - Z[data_RS2_0[:,1,:].astype(int)], 0), axis=0)),
    np.cumsum(np.mean(np.maximum(tau - kappa_rs1*epsilon_t - Z[data_RSG_0[:,1,:].astype(int)], 0), axis=0)),
    np.cumsum(np.mean(np.maximum(tau - kappa_rs1*epsilon_t - Z[data_RO_true_0[:,1,:].astype(int)], 0), axis=0)),
    np.cumsum(np.mean(np.maximum(tau - kappa_rs1*epsilon_t - Z[data_RO_small_0[:,1,:].astype(int)], 0), axis=0)),
    np.cumsum(np.mean(np.maximum(tau - kappa_rs1*epsilon_t - Z[data_RO_large_0[:,1,:].astype(int)], 0), axis=0))
]
cum_rs_reg_list_1 = [
    np.cumsum(np.mean(np.maximum(tau - kappa_rs1*epsilon_t - Z[data_RS1_1[:,1,:].astype(int)], 0), axis=0)),
    np.cumsum(np.mean(np.maximum(tau - kappa_rs1*epsilon_t - Z[data_RS2_1[:,1,:].astype(int)], 0), axis=0)),
    np.cumsum(np.mean(np.maximum(tau - kappa_rs1*epsilon_t - Z[data_RSG_1[:,1,:].astype(int)], 0), axis=0)),
    np.cumsum(np.mean(np.maximum(tau - kappa_rs1*epsilon_t - Z[data_RO_true_1[:,1,:].astype(int)], 0), axis=0)),
    np.cumsum(np.mean(np.maximum(tau - kappa_rs1*epsilon_t - Z[data_RO_small_1[:,1,:].astype(int)], 0), axis=0)),
    np.cumsum(np.mean(np.maximum(tau - kappa_rs1*epsilon_t - Z[data_RO_large_1[:,1,:].astype(int)], 0), axis=0))
]
cum_rs_reg_list_2 = [
    np.cumsum(np.mean(np.maximum(tau - kappa_rs1*epsilon_t - Z[data_RS1_2[:,1,:].astype(int)], 0), axis=0)),
    np.cumsum(np.mean(np.maximum(tau - kappa_rs1*epsilon_t - Z[data_RS2_2[:,1,:].astype(int)], 0), axis=0)),
    np.cumsum(np.mean(np.maximum(tau - kappa_rs1*epsilon_t - Z[data_RSG_2[:,1,:].astype(int)], 0), axis=0)),
    np.cumsum(np.mean(np.maximum(tau - kappa_rs1*epsilon_t - Z[data_RO_true_2[:,1,:].astype(int)], 0), axis=0)),
    np.cumsum(np.mean(np.maximum(tau - kappa_rs1*epsilon_t - Z[data_RO_small_2[:,1,:].astype(int)], 0), axis=0)),
    np.cumsum(np.mean(np.maximum(tau - kappa_rs1*epsilon_t - Z[data_RO_large_2[:,1,:].astype(int)], 0), axis=0))
]

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

# Plotting cum_reg_list_0
axs[0].plot(cum_rs_reg_list_0[0], label='RS1', c='r', linestyle='--', marker='s', markevery=10)
axs[0].plot(cum_rs_reg_list_0[1], label='RS2', c='m', linestyle='--', marker='s', markevery=10)
axs[0].plot(cum_rs_reg_list_0[2], label='RSG', c='orange', linestyle='--', marker='s', markevery=10)
axs[0].plot(cum_rs_reg_list_0[3], label=r'RO $r=\epsilon_t$', c='b', linestyle='--', marker='s', markevery=10)
axs[0].plot(cum_rs_reg_list_0[4], label=r'RO $r=0.5 \cdot \epsilon_t$', c='g', linestyle='--', marker='s', markevery=10)
axs[0].plot(cum_rs_reg_list_0[5], label=r'RO $r=3 \cdot \epsilon_t$', c='darkred', linestyle='--', marker='s', markevery=10)
axs[0].set_xlabel('Round t')
axs[0].set_ylabel('Cumulative Regret')
axs[0].set_title('Random Attack')

# Plotting cum_reg_list_1
axs[1].plot(cum_rs_reg_list_1[0], label='RS1', c='r', linestyle='--', marker='s', markevery=10)
axs[1].plot(cum_rs_reg_list_1[1], label='RS2', c='m', linestyle='--', marker='s', markevery=10)
axs[1].plot(cum_rs_reg_list_1[2], label='RSG', c='orange', linestyle='--', marker='s', markevery=10)
axs[1].plot(cum_rs_reg_list_1[3], label=r'RO $r=\epsilon_t$', c='b', linestyle='--', marker='s', markevery=10)
axs[1].plot(cum_rs_reg_list_1[4], label=r'RO $r=0.5 \cdot \epsilon_t$', c='g', linestyle='--', marker='s', markevery=10)
axs[1].plot(cum_rs_reg_list_1[5], label=r'RO $r=3 \cdot \epsilon_t$', c='darkred', linestyle='--', marker='s', markevery=10)
axs[1].set_xlabel('Round t')
axs[1].set_ylabel('Cumulative Regret')
axs[1].set_title('LCB Attack')

# Plotting cum_reg_list_2
axs[2].plot(cum_rs_reg_list_2[0], label='RS1', c='r', linestyle='--', marker='s', markevery=10)
axs[2].plot(cum_rs_reg_list_2[1], label='RS2', c='m', linestyle='--', marker='s', markevery=10)
axs[2].plot(cum_rs_reg_list_2[2], label='RSG', c='orange', linestyle='--', marker='s', markevery=10)
axs[2].plot(cum_rs_reg_list_2[3], label=r'RO $r=\epsilon_t$', c='b', linestyle='--', marker='s', markevery=10)
axs[2].plot(cum_rs_reg_list_2[4], label=r'RO $r=0.5 \cdot \epsilon_t$', c='g', linestyle='--', marker='s', markevery=10)
axs[2].plot(cum_rs_reg_list_2[5], label=r'RO $r=3 \cdot \epsilon_t$', c='darkred', linestyle='--', marker='s', markevery=10)
axs[2].set_xlabel('Round t')
axs[2].set_ylabel('Cumulative Regret')
axs[2].set_title('Worst Case Attack')

# Adding legend and adjusting layout
axs[0].legend()
axs[1].legend()
axs[2].legend()
# Adding legends for tau and epsilon_t
axs[0].legend(title=r'$\tau=1$, $\epsilon_t=0.1$')
axs[1].legend(title=r'$\tau=1$, $\epsilon_t=0.1$')
axs[2].legend(title=r'$\tau=1$, $\epsilon_t=0.1$')

plt.tight_layout()
# Save as pdf
dir = 'figures'
os.makedirs(dir, exist_ok=True)
# plt.savefig(os.path.join(dir, 'regret_GP_func_N_100.pdf'), format='pdf', bbox_inches='tight')
# Display the plot
plt.show()